# Symbolic AI History

## Expert Systems, Prolog, and the First AI Winter (Narrative)

Symbolic AI dominated from 1956-1980. The central claim: intelligence is symbol manipulation according to explicit rules. Key milestones:

- **LISP (1958)**: McCarthy's language for AI — list processing, recursion, symbolic manipulation
- **Logic Theorist (1956)**: First AI program, proved 38/52 theorems from Principia Mathematica
- **MYCIN (1972-1976)**: Medical expert system, 450+ if-then rules for bacterial infections, outperformed junior physicians
- **Prolog (1972)**: Logic programming language. Programs are facts + rules; execution is theorem proving via unification and backtracking
- **Cyc (1984-present)**: Lenat's attempt to encode all human common sense, 25M facts manually authored

**The First AI Winter (1974-1980)**: Lighthill report (UK), DARPA funding cuts. Symbolic AI failed to scale: combinatorial explosion in search, frame problem (how to represent what *doesn't* change), brittleness to out-of-distribution inputs.

**Inductive Logic Programming (ILP)**: Learn Prolog-style rules from examples. Successful in molecular biology (Muggleton, 1991) but O(n²) or worse in worst case.

## Why Modern Neurosymbolic AI
Neural networks solve perception and pattern recognition; symbolic systems handle reasoning and compositionality. Neurosymbolic AI combines both — neural perception feeding into symbolic reasoning, or symbolic constraints guiding neural optimization.

In [ ]:
import json
from typing import Dict, List, Set, Optional, Tuple

# Modern knowledge representation using Python
# In production: use networkx (graph) or owlready2 (OWL ontologies)

class KnowledgeGraph:
    """Simple triple-store for knowledge representation."""
    
    def __init__(self):
        self.triples: List[Tuple[str, str, str]] = []  # (subject, predicate, object)
        self.entities: Set[str] = set()
        self.relations: Dict[str, List[Tuple[str, str]]] = {}
    
    def add(self, subject: str, predicate: str, obj: str):
        self.triples.append((subject, predicate, obj))
        self.entities.update([subject, obj])
        if predicate not in self.relations:
            self.relations[predicate] = []
        self.relations[predicate].append((subject, obj))
        return self
    
    def query(self, subject=None, predicate=None, obj=None) -> List[Tuple]:
        results = []
        for s, p, o in self.triples:
            if ((subject is None or s == subject) and
                (predicate is None or p == predicate) and
                (obj is None or o == obj)):
                results.append((s, p, o))
        return results
    
    def forward_chain(self, rules: List[Dict], max_iterations: int = 10) -> int:
        """Apply inference rules to derive new facts."""
        new_facts = 0
        for _ in range(max_iterations):
            added = 0
            for rule in rules:
                # Rule: if (?, pred1, X) and (X, pred2, ?) -> (?, pred3, ?)
                for premise in rule['premises']:
                    matches = self.query(predicate=premise[1])
                    for s, p, o in matches:
                        # Apply conclusion
                        conclusion = rule['conclusion']
                        new_s = s if conclusion[0] == '?' else conclusion[0]
                        new_o = o if conclusion[2] == '?' else conclusion[2]
                        new_p = conclusion[1]
                        new_triple = (new_s, new_p, new_o)
                        if new_triple not in [(t[0], t[1], t[2]) for t in self.triples]:
                            self.triples.append(new_triple)
                            self.entities.update([new_s, new_o])
                            added += 1
            new_facts += added
            if added == 0:
                break
        return new_facts

# Build a small medical knowledge graph
kg = KnowledgeGraph()

# Facts
kg.add('pneumonia', 'is_a', 'bacterial_infection')
kg.add('bacterial_infection', 'treated_by', 'antibiotics')
kg.add('penicillin', 'is_a', 'antibiotics')
kg.add('amoxicillin', 'is_a', 'antibiotics')
kg.add('patient_X', 'diagnosed_with', 'pneumonia')
kg.add('strep_throat', 'is_a', 'bacterial_infection')
kg.add('patient_Y', 'diagnosed_with', 'strep_throat')

print(f'Knowledge graph: {len(kg.triples)} triples, {len(kg.entities)} entities')
print('\nDirect query: What treats bacterial infections?')
for s, p, o in kg.query(subject='bacterial_infection', predicate='treated_by'):
    print(f'  {s} --[{p}]--> {o}')

print('\nQuery: What is patient_X diagnosed with?')
for s, p, o in kg.query(subject='patient_X'):
    print(f'  {s} --[{p}]--> {o}')
